<h1>Silver Bus Live Feed<h1>

<h2>Imports<h2>

In [16]:
import os
import sys
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from config import DB_CONFIG
import hashlib
from google.transit import gtfs_realtime_pb2

<h2>Bus Live Feed Table<h2>

<h3>Calling GTFS-Realtime Protobuf API and Creating DataFrame<h3>

In [17]:
url = "https://mkuran.pl/gtfs/warsaw/vehicles.pb"
engine = create_engine(f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")

In [18]:
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)
    dane_autobusow = [
        (
            e.vehicle.vehicle.id,
            e.vehicle.trip.trip_id,
            e.vehicle.trip.route_id,
            e.vehicle.position.latitude,
            e.vehicle.position.longitude,
            e.vehicle.timestamp
        )
        for e in feed.entity
        if e.HasField('vehicle') and e.vehicle.trip.trip_id
    ]
    df_rt = pd.DataFrame(dane_autobusow, columns=[
        "vehicle_number", "trip_id", "route_id", "lat", "lon", "time_gps_raw"
    ])
except Exception as e:
    print(f"Error: {e}")
if df_rt.empty:
    sys.exit(99)

<h3>Cleaning and Transforming Data<h3>

In [19]:
df_rt['time_gps'] = pd.to_datetime(df_rt['time_gps_raw'], unit='s')
df_rt = df_rt.drop(columns=['time_gps_raw', 'route_id'])
df_rt['etl_timestamp'] = pd.Timestamp.now()

In [20]:
hash_input = df_rt['vehicle_number'].astype(str) + '_' + df_rt['time_gps'].astype(str)
df_rt['gps_id'] = [hashlib.md5(x.encode('utf-8')).hexdigest() for x in hash_input]

In [21]:
df_rt = df_rt[['gps_id', 'trip_id', 'lat', 'lon', 'vehicle_number', 'time_gps', 'etl_timestamp']]

<h3>Writing to Database<h3>

In [ ]:
df_rt.to_sql('bus_live_feed', engine, schema='silver', if_exists='append', index=False)

272